## Logistic Regression with PySpark: Customer Churn

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName('churn').getOrCreate()
spark.conf.set("spark.sql.debug.maxToStringFields", 1000)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/03 09:46:48 WARN Utils: Your hostname, aditya-HP-Laptop-15s-eq1xxx, resolves to a loopback address: 127.0.1.1; using 10.103.210.123 instead (on interface wlo1)
26/07/03 09:46:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/03 09:46:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
!curl https://raw.githubusercontent.com/markumreed/data_science_for_everyone/refs/heads/main/pyspark_examples/data/customer_churn.csv >> customer_churn.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  112k  100  112k    0     0   156k      0 --:--:-- --:--:-- --:--:--  156k


In [4]:
df = spark.read.csv('customer_churn.csv', inferSchema=True, header=True)

In [5]:
df.printSchema()

root
 |-- Names: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- Total_Purchase: double (nullable = true)
 |-- Account_Manager: integer (nullable = true)
 |-- Years: double (nullable = true)
 |-- Num_Sites: double (nullable = true)
 |-- Onboard_date: timestamp (nullable = true)
 |-- Location: string (nullable = true)
 |-- Company: string (nullable = true)
 |-- Churn: integer (nullable = true)



In [10]:
df.describe().toPandas()

,summary,Names,Age,Total_Purchase,Account_Manager,Years,Num_Sites,Location,Company,Churn
0,count,2702,2700,2700,2700,2700,2700,2702,2702,2700
1,mean,None,41.81666666666667,10062.824033333369,0.4811111111111111,5.273155555555563,8.587777777777777,None,None,0.16666666666666666
2,stddev,None,6.125289688501455,2407.751945378731,0.49973563469292864,1.2739767326472888,1.7641815858640209,None,None,0.37274702987031233
3,min,Aaron King,22.0,100.0,0,1.0,3.0,"00103 Jeffrey Crest Apt. 205 Padillaville, IA ...",Abbott-Thompson,0
4,max,Zachary Walsh,65.0,18026.01,1,9.15,14.0,Unit 9800 Box 2878 DPO AA 75157,"Zuniga, Clark and Shaffer",1


In [7]:
df.columns

['Names',
 'Age',
 'Total_Purchase',
 'Account_Manager',
 'Years',
 'Num_Sites',
 'Onboard_date',
 'Location',
 'Company',
 'Churn']

In [8]:
from pyspark.ml.feature import VectorAssembler

In [19]:
assembler = VectorAssembler(inputCols=[
     'Age',
     'Total_Purchase',
     'Account_Manager',
     'Years',
     'Num_Sites'],
    outputCol='features', 
    handleInvalid='skip')

In [20]:
output = assembler.transform(df)

In [21]:
df_final = output.select('features', 'churn')

In [22]:
df_final.show()

+--------------------+-----+
|            features|churn|
+--------------------+-----+
|[42.0,11066.8,0.0...|    1|
|[41.0,11916.22,0....|    1|
|[38.0,12884.75,0....|    1|
|[42.0,8010.76,0.0...|    1|
|[37.0,9191.58,0.0...|    1|
|[48.0,10356.02,0....|    1|
|[44.0,11331.58,1....|    1|
|[32.0,9885.12,1.0...|    1|
|[43.0,14062.6,1.0...|    1|
|[40.0,8066.94,1.0...|    1|
|[30.0,11575.37,1....|    1|
|[45.0,8771.02,1.0...|    1|
|[45.0,8988.67,1.0...|    1|
|[40.0,8283.32,1.0...|    1|
|[41.0,6569.87,1.0...|    1|
|[38.0,10494.82,1....|    1|
|[45.0,8213.41,1.0...|    1|
|[43.0,11226.88,0....|    1|
|[53.0,5515.09,0.0...|    1|
|[46.0,8046.4,1.0,...|    1|
+--------------------+-----+
only showing top 20 rows


In [23]:
train, test = df_final.randomSplit([0.7, 0.3], seed=42)

In [24]:
from pyspark.ml.classification import LogisticRegression

In [25]:
lr = LogisticRegression(labelCol='churn')

In [26]:
lrm = lr.fit(train)

In [27]:
lrm.summary

In [28]:
lrm_summary = lrm.summary

In [29]:
lrm_summary.predictions.show()

+--------------------+-----+--------------------+--------------------+----------+
|            features|churn|       rawPrediction|         probability|prediction|
+--------------------+-----+--------------------+--------------------+----------+
|[22.0,11254.38,1....|  0.0|[4.88093872051691...|[0.99246728653489...|       0.0|
|[22.0,11254.38,1....|  0.0|[4.88093872051691...|[0.99246728653489...|       0.0|
|[25.0,9672.03,0.0...|  0.0|[4.95551718481787...|[0.99300484088913...|       0.0|
|[25.0,9672.03,0.0...|  0.0|[4.95551718481787...|[0.99300484088913...|       0.0|
|[25.0,9672.03,0.0...|  0.0|[4.95551718481787...|[0.99300484088913...|       0.0|
|[26.0,8787.39,1.0...|  1.0|[0.75939067138654...|[0.68122142733583...|       0.0|
|[26.0,8939.61,0.0...|  0.0|[6.68322114561359...|[0.99874982545607...|       0.0|
|[26.0,8939.61,0.0...|  0.0|[6.68322114561359...|[0.99874982545607...|       0.0|
|[27.0,8628.8,1.0,...|  0.0|[5.72362008032612...|[0.99674278457274...|       0.0|
|[28.0,8670.98,0

In [30]:
lrm_summary.predictions.describe().show()

+-------+-------------------+-------------------+
|summary|              churn|         prediction|
+-------+-------------------+-------------------+
|  count|               1937|               1937|
|   mean|0.16520392359318534|0.12803304078471864|
| stddev|0.37146039162551475| 0.3342128765347424|
|    min|                0.0|                0.0|
|    max|                1.0|                1.0|
+-------+-------------------+-------------------+



In [31]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

In [32]:
pred_labels = lrm.evaluate(test)

In [33]:
pred_labels.predictions.show()

+--------------------+-----+--------------------+--------------------+----------+
|            features|churn|       rawPrediction|         probability|prediction|
+--------------------+-----+--------------------+--------------------+----------+
|[22.0,11254.38,1....|    0|[4.88093872051691...|[0.99246728653489...|       0.0|
|[26.0,8787.39,1.0...|    1|[0.75939067138654...|[0.68122142733583...|       0.0|
|[26.0,8787.39,1.0...|    1|[0.75939067138654...|[0.68122142733583...|       0.0|
|[26.0,8939.61,0.0...|    0|[6.68322114561359...|[0.99874982545607...|       0.0|
|[27.0,8628.8,1.0,...|    0|[5.72362008032612...|[0.99674278457274...|       0.0|
|[27.0,8628.8,1.0,...|    0|[5.72362008032612...|[0.99674278457274...|       0.0|
|[28.0,8670.98,0.0...|    0|[8.10088688569187...|[0.99969682189193...|       0.0|
|[28.0,9090.43,1.0...|    0|[1.67263396907107...|[0.84192668173368...|       0.0|
|[28.0,11128.95,1....|    0|[4.39052187058708...|[0.98775747766375...|       0.0|
|[28.0,11128.95,

In [34]:
evals = BinaryClassificationEvaluator(rawPredictionCol='prediction', labelCol='churn')

In [35]:
auc = evals.evaluate(pred_labels.predictions)

In [36]:
auc

0.7411046299671892

In [38]:
customers = spark.read.csv('new_customers.csv', inferSchema=True, header=True)

26/07/03 10:07:34 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: new_customers.csv.
java.io.FileNotFoundException: File new_customers.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.ResolveData

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/aditya/DataScience/data-science-for-everyone/pyspark/new_customers.csv. SQLSTATE: 42K03